# P2 — RepDNet / RepDWNet + WWSCE training on Kaggle

This notebook trains the P2 experiment based on the P1_EliPRD (former B1 Run 5) recipe: four DW 3×3 branches and Weed Size-Aware Weighted Cross-Entropy. Enable **Internet** and attach the Kaggle PhenoBench dataset before running.

The source repository must already contain the committed WWSCE implementation on `f/wwsce_loss`. The final cell produces one ZIP containing checkpoints, logs, resolved configuration, quantile artifact, and source revision manifest.

In [ ]:
# Kaggle runtime dependencies. Kaggle supplies torch/CUDA; do not reinstall torch.
!nvidia-smi
!pip install -q --upgrade --no-deps 'pytorch-lightning==1.9.5' 'torchmetrics==0.11.4'
!pip install -q 'setuptools<70' tensorboard matplotlib pandas tabulate timm scikit-image tifffile albumentations pyyaml rich


In [ ]:
# ---- User settings ---------------------------------------------------------
# Set DATASET_DIR only when automatic discovery does not find your Kaggle input.
REPO_URL = 'https://github.com/FUAVX/RPD_Reproduce.git'
GIT_REF = 'f/wwsce_loss'  # pin to a commit SHA after the branch has been pushed
SCENARIO = 'P2_repdnet_wwsce_run5'
DATASET_DIR = ''  # e.g. '/kaggle/input/<your-phenobench-slug>/PhenoBench'
NUM_WORKERS = 2

import json
import os
import pathlib
import shutil
import subprocess
import yaml

WORKING_DIR = pathlib.Path('/kaggle/working')
REPO_DIR = WORKING_DIR / 'RPD_P2'
RUN_DIR = WORKING_DIR / 'p2_repdnet_wwsce_run5'

def run(command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run(['git', 'clone', '--branch', GIT_REF, '--single-branch', REPO_URL, str(REPO_DIR)])
run(['git', 'rev-parse', '--verify', 'HEAD'], cwd=REPO_DIR)

scenario_path = REPO_DIR / 'configs' / 'p2_scenarios' / f'{SCENARIO}.yaml'
quantile_path = REPO_DIR / 'experiments' / 'p2_weed_visible_area_quantiles.json'
required_paths = [scenario_path, quantile_path, REPO_DIR / 'test_wwsce.py']
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Branch does not contain the committed P2 WWSCE files: ' + ', '.join(missing))

def is_phenobench_root(path):
    return (path / 'train' / 'images').is_dir() and (path / 'train' / 'annotations').is_dir() and (path / 'train' / 'plant_instances').is_dir()

if DATASET_DIR:
    dataset_dir = pathlib.Path(DATASET_DIR)
else:
    candidates = [pathlib.Path('/kaggle/input')]
    dataset_dir = next((p for p in candidates[0].rglob('*') if p.is_dir() and is_phenobench_root(p)), None)
if dataset_dir is None or not is_phenobench_root(dataset_dir):
    raise FileNotFoundError('Cannot find a complete PhenoBench root (train/images, train/annotations, train/plant_instances). Set DATASET_DIR in this cell.')

print(f'[OK] source ref: {subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()}')
print(f'[OK] dataset: {dataset_dir}')
print(f'[OK] train images: {len(list((dataset_dir / "train" / "images").glob("*")))}')


In [ ]:
# Resolve the Kaggle-specific configuration without modifying the cloned repository.
RUN_DIR.mkdir(parents=True, exist_ok=True)
with scenario_path.open(encoding='utf-8') as stream:
    cfg = yaml.safe_load(stream)

cfg['data']['path_to_dataset'] = str(dataset_dir)
cfg['data']['num_workers'] = NUM_WORKERS
# Preserve provenance even though the training loader reads the numeric values from YAML.
cfg['loss']['threshold_artifact'] = str(quantile_path)
active_cfg_path = RUN_DIR / f'active_config_{SCENARIO}.yaml'
with active_cfg_path.open('w', encoding='utf-8') as stream:
    yaml.safe_dump(cfg, stream, sort_keys=False)

assert cfg['train']['loss'] == 'weed_size_aware_ce'
assert cfg['loss']['weed_size']['q25'] == 127
assert cfg['loss']['weed_size']['q75'] == 776
print(yaml.safe_dump({
    'experiment': cfg['experiment'],
    'backbone': cfg['backbone'],
    'loss': cfg['loss'],
}, sort_keys=False))


In [ ]:
# Optional quick check, then train with Kaggle T4 x2 (DDP), matching the teammate workflow.
run(['python', 'test_wwsce.py'], cwd=REPO_DIR)

TRAIN_LOG_DIR = RUN_DIR / 'training_output'
run([
    'python', 'multi_metric_train.py',
    '--config', str(active_cfg_path),
    '--export_dir', str(TRAIN_LOG_DIR),
    '--devices', '2',
], cwd=REPO_DIR)


In [ ]:
# Package every artifact needed for evaluation and later analysis.
source_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
manifest = {
    'experiment': SCENARIO,
    'source_repository': REPO_URL,
    'requested_ref': GIT_REF,
    'source_commit': source_commit,
    'dataset_dir_at_runtime': str(dataset_dir),
    'quantiles': cfg['loss']['weed_size'],
}
(RUN_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
shutil.copy2(quantile_path, RUN_DIR / quantile_path.name)
archive = shutil.make_archive(str(WORKING_DIR / f'{SCENARIO}_artifacts'), 'zip', root_dir=RUN_DIR)
print(f'[DONE] Download this Kaggle output: {archive}')
print(f'[DONE] Source commit: {source_commit}')
